In [ ]:
# python -m venv voi-aug25
# source voi-aug25/bin/activate
# pip install --index-url https://nexus2.discoverfinancial.com/repository/pypi-all/simple --trusted-host nexus2.discoverfinancial.com ipykernel
# python -m ipykernel install --user --name="voi-aug25"
# pip install -U --index-url https://nexus2.discoverfinancial.com/repository/pypi-all/simple --trusted-host nexus2.discoverfinancial.com nvidia-cufft-cu12==11.2.1.3 torch==2.6.0 transformers==4.52.4 accelerate==1.7.0

In [ ]:
# IMPORTS
import re
import os
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_mutual_info_score
from scipy.optimize import linear_sum_assignment
#from kmodes.kmodes import KModes
from kmodes.kprototypes import KPrototypes
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
LAST_MONTH_RAW = "X.csv"
THIS_MONTH_RAW = "Y.csv"
LAST_CLUSTER_COL = "Cluster"

In [ ]:
CATEGORICAL_DUMMY_COLS = ['Business Unit','Level', 'FLSA', 'Tenure']
NUMERIC_COLS = ['At work, I feel cared about as a person.', 'I am excited about being part of Capital One.', 'The information I receive about the integration is helpful.', 
    'I can trust my senior leaders.', 'My morale is good.'
]
SAVE_DIR = Path("SAVE_PATH_HERE"); SAVE_DIR.mkdir(parents=True, exist_ok=True)
#OLD_KMODES_PATH= SAVE_DIR / "kmode_voice_model.pkl"

In [ ]:
K_NEW = 6
KPROTO_INIT = "Cao"
K_PROTO_N_INIT = 5
KPROTO_RANDOM_STATE = 42

In [ ]:
# Weak fit thresholds (in both spaces)
ROW_NOVELTY_TOP_Q = 0.10
ADD_KPROTO_WEAKFIT = True

In [ ]:
# ALIGNMENT WEIGHTS
W_CAT = 1.0 # Jensen-Shannon for categorical distributions
W_NUM = 1.0 # standardized mean gaps
EPS = 1e-12

In [ ]:
# K + 1 emergent prob
KPLUS_CHECK = True
NEW_CLUSTER_MIN_PROP = 0.08
NEW_CLUSTER_MAX_MATCH_DIST = 0.35

In [ ]:
# HELPERS
def prep_raw(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in CATEGORICAL_DUMMY_COLS:
        df[c] = df[c].astype("string").fillna("__NA__")
    for c in NUMERIC_COLS:
        df[c] = pd.to_numeric(df[c], errors = "coerce")
        med = df[c].median()
        df[c] = df[c].fillna(med if pd.notna(med) else 0.0)
    return df

In [ ]:
def jsd(p: pd.Series, q: pd.Series, eps=1e-12) -> float:
    cats = p.index.union(q.index)
    p = p.reindex(cats, fill_value=0.0).astype(float)
    q = q.reindex(cats, fill_value=0.0).astype(float)
    m = 0.5*(p+q)
    def _kl(a,b):
        a = a.clip(eps, 1.0); b = b.clip(eps,1.0)
        return float(np.sum(a*np.log(a/b)))
    return 0.5*_kl(p,m) + 0.5*_kl(q,m)

In [ ]:
def cluster_profiles(df_raw: pd.DataFrame, labels: np.ndarray):
    out_cat, out_num = {}, {}
    x = df_raw.assign(_cl = labels)
    for cl in sorted(np.unique(labels)):
        sub = x.loc[x["_cl"] == cl]
        out_cat[cl] = {c: sub[c].value_counts(normalize=True, dropna=False) for c in CATEGORICAL_DUMMY_COLS}
        out_num[cl] = {c: {
            "mean": float(sub[c].mean()),
            "std": float(sub[c].std(ddof=1)) if len(sub) > 1 else np.nan
        } for c in NUMERIC_COLS}
    return out_cat, out_num

In [ ]:
def build_profile_distance_matrix(catA, numA, catB, numB, w_cat=1.0, w_num=1.0):
    A_ids, B_ids = sorted(catA.keys()), sorted(catB.keys())
    D = np.zeros((len(A_ids), len(B_ids)), dtype = float)
    for i,a in enumerate(A_ids):
        for j,b in enumerate(B_ids):
            dcat = np.mean([jsd(catA[a][c], catB[b][c]) for c in CATEGORICAL_DUMMY_COLS]) if CATEGORICAL_DUMMY_COLS else 0.0
            gaps = []
            for c in NUMERIC_COLS:
                muA, sdA = numA[a][c]["mean"], numA[a][c]["std"]
                muB, sdB = numB[b][c]["mean"], numB[b][c]["std"]
                sdA = 0.0 if np.isnan(sdA) else sdA
                sdB = 0.0 if np.isnan(sdB) else sdB
                denom = np.sqrt(sdA**2 + sdB**2) + EPS
                gaps.append(abs(muA - muB)/denom)
            dnum = np.mean(gaps) if NUMERIC_COLS else 0.0
            D[i,j] = w_cat*dcat + w_num*dnum

    # final guard to ensure Hungarian sees finite values
    if not np.isfinite(D).all():
        D = np.nan_to_num(D, nan = 1e9, posinf=1e9, neginf=1e9)
    return D, A_ids, B_ids

In [ ]:
def k_proto_row_distance_aligned(df_mixed, labels, kproto, cat_idx):
    n_cols = df_mixed.shape[1]
    all_idx = list(range(n_cols))
    num_idx = [i for i in all_idx if i not in cat_idx]
    num_count, cat_count = len(num_idx), len(cat_idx)

    X = df_mixed.to_numpy()
    C = kproto.cluster_centroids_
    gamma = kproto.gamma
    
    dists = np.zeros(len(df_mixed), dtype= float)
    for r, cl in enumerate(labels):
        c = C[cl]
        c_num = c[:num_count].astype(float)
        c_cat = c[num_count:num_count+cat_count]
        x_num = X[r, num_idx].astype(float)
        x_cat = X[r, cat_idx]
        ham = np.sum(x_cat != c_cat)
        eu2 = np.sum((x_num - c_num) **2)
        dists[r] = ham + gamma * eu2
    return dists
        

In [ ]:
# LOAD & PREP RAW DATA
df_last = prep_raw(pd.read_csv(LAST_MONTH_RAW, low_memory = False))
df_this = prep_raw(pd.read_csv(THIS_MONTH_RAW, low_memory = False))
if LAST_CLUSTER_COL not in df_last.columns:
    raise ValueError(f"Expected{LAST_CLUSTER_COL} in last survey's raw data & not found")

if "Employee ID" not in df_last.columns or "Employee ID" not in df_this.columns:
    raise ValueError("Both last and this month's data require an Employee ID")

df_last["Employee ID"] = df_last["Employee ID"].astype("string").str.strip()
df_this["Employee ID"] = df_this["Employee ID"].astype("string").str.strip()

In [ ]:
# CONTINUITY

#labels_last = df_last[LAST_CLUSTER_COL].to_numpy()
labels_last_raw = df_last[LAST_CLUSTER_COL]
labels_last_float = labels_last_raw.astype("float").to_numpy()
valid_mask = ~pd.isna(labels_last_float)
labels_last_valid = labels_last_float[valid_mask].astype(int)
unique_clusters = np.array(sorted(pd.unique(labels_last_valid)))

X_last_dum_df = pd.get_dummies(df_last.loc[valid_mask, CATEGORICAL_DUMMY_COLS],drop_first = False)
dummy_cols = list(X_last_dum_df.columns)

def _binary_mode(col: pd.Series) -> int:
    ones = int(col.sum()); zeros = int(len(col) - ones)
    return 1 if ones > zeros else 0
    
centroids = []
for k in unique_clusters:
    rows_k = (labels_last_valid == k)
    Ck = X_last_dum_df.loc[rows_k].astype(int).apply(_binary_mode, axis = 0).to_numpy(dtype=int)
    centroids.append(Ck)
centroids = np.vstack(centroids)

#  THIS MONTH'S DUMMIES TO REINDEX TO LAST MONTHS DUMMY COLUMNS
X_this_dum_df = pd.get_dummies(df_this[CATEGORICAL_DUMMY_COLS],drop_first = False)
for c in dummy_cols:
    if c not in X_this_dum_df.columns: 
        X_this_dum_df[c] = 0
X_this_dum = X_this_dum_df[dummy_cols].astype(int).to_numpy()

D_ham = np.abs(X_this_dum[:, None, :] - centroids[None, :, :]).sum(axis=2)
labels_cont_this = unique_clusters[D_ham.argmin(axis=1)].astype(int)
dist_old = D_ham.min(axis=1).astype(float)
thr_old = np.quantile(dist_old, 1 - ROW_NOVELTY_TOP_Q)
weakfit_old_flag = (dist_old >= thr_old).astype(int)

prev_feat = pd.Series(labels_cont_this, index = df_this.index).astype("Int64").astype("string")
prev_feat_masked = np.where(weakfit_old_flag == 1, "__UNSURE__", prev_feat)
df_this["PrevCluster_FromContinuity"]= prev_feat_masked

def _sizes_to(arr_labels, ids_sorted):
    return pd.Series(arr_labels).value_counts().reindex(ids_sorted, fill_value=0).sort_index()
    
sizes_last = _sizes_to(labels_last_valid, unique_clusters)
sizes_this = _sizes_to(labels_cont_this, unique_clusters)
cont_sizes = pd.DataFrame({
    "cluster_old_space": unique_clusters,
    "last_month_size": sizes_last.values,
    "this_month_pred_size": sizes_this.values
})
cont_sizes["delta"] = cont_sizes["this_month_pred_size"] - cont_sizes["last_month_size"]
cont_sizes.to_csv(SAVE_DIR / "continuity_sizes.csv", index = False)

In [ ]:
# NOVELTY 
# K-PROTOTYPES ON THIS MONTH'S RAW MIXED (CATS THEN NUMS)
cats_df = df_this[CATEGORICAL_DUMMY_COLS].copy()
nums_df = df_this[NUMERIC_COLS].copy()

for c in CATEGORICAL_DUMMY_COLS:
    cats_df[c] = cats_df[c].astype("string").fillna("__NA__")
for c in NUMERIC_COLS:
    nums_df[c] = pd.to_numeric(nums_df[c], errors = "coerce")
    nums_df[c] = nums_df[c].fillna(nums_df[c].median())

cats_df["PrevCluster_FromContinuity"] = (
    df_this["PrevCluster_FromContinuity"]
    .astype("string")
    .fillna("__UNSURE__")
)

# build mixed matrix with new categorical at the end (for clarity)
CATS_FOR_KP = CATEGORICAL_DUMMY_COLS + ["PrevCluster_FromContinuity"]
row_ids = df_this["Employee ID"].copy()
this_mixed = pd.concat([cats_df[CATS_FOR_KP], nums_df], axis = 1)

# categorical positions by name
cat_idx = [this_mixed.columns.get_loc(c) for c in CATS_FOR_KP]

# FIT KPROTOTYPES
kproto = KPrototypes(n_clusters = K_NEW, 
                     init = KPROTO_INIT, 
                     n_init = K_PROTO_N_INIT, 
                     random_state = KPROTO_RANDOM_STATE, 
                     verbose = 0)
labels_kp = kproto.fit_predict(this_mixed.to_numpy(), categorical = cat_idx)

if ADD_KPROTO_WEAKFIT:
    dist_kp = k_proto_row_distance_aligned(this_mixed, labels_kp, kproto, cat_idx)
    thr_kp = np.quantile(dist_kp, 1 - ROW_NOVELTY_TOP_Q)
    weakfit_kp_flag = (dist_kp >= thr_kp).astype(int)
else:
    dist_kp = np.zeros(len(this_mixed), dtype=float)
    weakfit_kp_flag = np.zeros(len(this_mixed), dtype=int)


pd.Series(labels_kp).value_counts().sort_index().rename("size").reset_index().rename(
    columns= {"index": "kproto_cluster"}).to_csv(SAVE_DIR / "kproto_sizes.csv", index = False)

In [ ]:
# ALIGNMENT  
# CONTINUITY (OLD SPACE) VS. KPROTO (MIXED)
base_for_align = this_mixed
present_kp = sorted(np.unique(labels_kp).tolist()) # which cluster IDs are actually present
present_cont = sorted(np.unique(labels_cont_this).tolist())

catA, numA = cluster_profiles(base_for_align, labels_cont_this) # cont. labels on raw
catB, numB = cluster_profiles(base_for_align, labels_kp) # kproto labels on raw

D, A_ids, B_ids = build_profile_distance_matrix(catA, numA, catB, numB, w_cat = W_CAT, w_num = W_NUM)
ri, cj = linear_sum_assignment(D)

# map only present clusters
mapping_kp_to_cont = {}
for i, j in zip(ri, cj):
    a_id, b_id = A_ids[i], B_ids[j]
    if pd.notna(a_id) and pd.notna(b_id):
        mapping_kp_to_cont[int(b_id)] = int(a_id)

missing_kp_ids = sorted(set(present_kp) - set(mapping_kp_to_cont.keys()))
all_kp_ids = set(range(K_NEW))
never_seen_ids = sorted(all_kp_ids - set(present_kp))

labels_kp_aligned = np.array([mapping_kp_to_cont.get(int(c), -1) for c in labels_kp])

rows_align = []
for i, j in zip(ri, cj):
    a_id, b_id = A_ids[i], B_ids[j]
    if pd.notna(a_id) and pd.notna(b_id):
        rows_align.append({
            "continuity_cluster": int(a_id),
            "kproto_cluster": int(b_id),
            "match_distance": float(D[i, j])
        })

alignment_df = pd.DataFrame(rows_align).sort_values("continuity_cluster")
alignment_df.to_csv(SAVE_DIR / "continuity_vs_proto_alignment.csv", index=False)

mask_mapped = labels_kp_aligned != -1
ami = adjusted_mutual_info_score(labels_cont_this[mask_mapped], labels_kp_aligned[mask_mapped])

with open (SAVE_DIR / "agreement_metrics.txt", "w") as f:
    f.write(f"AMI (continuity KMODES vs. KPROTO aligned, mapped rows only): {ami:.4f}\n")
    f.write(f"Present k-proto clusters: {present_kp}\n")
    f.write(f"Missing (unmapped k-proto clusters in mapping: {missing_kp_ids}\n")
    f.write(f"Never-seen k-proto clusters (empty): {never_seen_ids}\n")

In [ ]:
# K+1 EMERGENT PROBE (KPROTOTYPES)
if KPLUS_CHECK:
    kproto_plus = KPrototypes(n_clusters = K_NEW + 1,
                              init = KPROTO_INIT, n_init = K_PROTO_N_INIT,
                              random_state = KPROTO_RANDOM_STATE, verbose = 0)
    labels_plus = kproto_plus.fit_predict(this_mixed.to_numpy(), categorical = cat_idx)

    catK, numK = cluster_profiles(this_mixed, labels_kp)
    catKp, numKp = cluster_profiles(this_mixed, labels_plus)

    Dk, idsK, idsKp = build_profile_distance_matrix(catK, numK, catKp, numKp, w_cat = W_CAT, w_num = W_NUM)
    r2, c2 = linear_sum_assignment(Dk)

    sizes_plus = pd.Series(labels_plus).value_counts()
    kplus_map = pd.DataFrame([{
        "kprotoK_cluster": int(idsK[i]),
        "kprotoKplus_cluster": int(idsKp[j]),
        "match_distance": float(Dk[i, j]),
        "kplus_size": int(sizes_plus.loc[idsKp[j]])
    } for i, j in zip(r2, c2)]).sort_values("match_distance")
    kplus_map.to_csv(SAVE_DIR / "kproto_K_vs_Kplus_map.csv", index = False)


    emergent_kplus_clusters = set(
        kplus_map.loc[
            (kplus_map["match_distance"] > NEW_CLUSTER_MAX_MATCH_DIST) & 
            (kplus_map["kplus_size"] >= NEW_CLUSTER_MIN_PROP * len(this_mixed)),
            "kprotoKplus_cluster"
            ].astype(int).tolist()
    )
else: 
    labels_plus = np.full(len(this_mixed), -1, dtype = int)
    emergent_kplus_clusters = set()

In [ ]:
# ROW LEVEL EXPORT
rows = this_mixed.copy()
rows.insert(0, "Employee ID", row_ids.values)

rows["Cluster_Continuity_OldSpace"] = labels_cont_this
rows["Cluster_KProto"] = labels_kp
rows["Cluster_KProto_AlignedToOld"] = labels_kp_aligned
rows["OldSpace_Hamming"] = dist_old
rows["WeakFit_OldSpace_Top{:02d}pct".format(int(ROW_NOVELTY_TOP_Q*100))] = (dist_old >= thr_old).astype(int)

if ADD_KPROTO_WEAKFIT:
    rows["KProto_MixedDistance"] = dist_kp
    rows["WeakFit_KProto_Top{:02d}pct".format(int(ROW_NOVELTY_TOP_Q*100))] = weakfit_kp_flag

rows["Cluster_KProto_Kplus"] = labels_plus
rows["KPlus_Emergent_Flag"] = rows["Cluster_KProto_Kplus"].isin(
    list(emergent_kplus_clusters)
).astype(int)

rows.to_csv(SAVE_DIR / "this_month_rows_with_labels.csv", index = False)
#cont_sizes.to_csv(SAVE_DIR / "continuity_sizes.csv", index = False) # already exported above

In [ ]:
# -----------------------------
# 4. Load RAG Context Documents
# -----------------------------
rag_folder = "documents" # Folder where RAG context documents are stored
raw_docs = []
for filename in os.listdir(rag_folder):
    if filename.endswith(".txt"):
        loader = TextLoader(os.path.join(rag_folder, filename), encoding='utf-8')
        raw_docs.extend(loader.load())

# split docs into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = splitter.split_documents(raw_docs)

# embed docs using HuggingFace MiniLM
embedding_model_path = ("PATH_TO_HUGGING_FACE_MINILM")
embeddings = SentenceTransformerEmbeddings(model_name=embedding_model_path)

db = FAISS.from_documents(docs, embeddings)
print(f"Total docs indexed: {db.index.ntotal}")

# RAG-style doc retrieval for a comment 
def retrieve_context(query, k=3):
    docs_and_scores = db.similarity_search_with_score(query, k=k)
    top_docs = [doc.page_content for doc, score in docs_and_scores]
    return "\n\n".join(top_docs)

In [ ]:
rows_tojoin = rows[['Employee ID', 'Cluster_Continuity_OldSpace', 'Cluster_KProto',
       'Cluster_KProto_AlignedToOld', 'OldSpace_Hamming',
       'WeakFit_OldSpace_Top10pct', 'KProto_MixedDistance',
       'WeakFit_KProto_Top10pct', 'Cluster_KProto_Kplus',
       'KPlus_Emergent_Flag']]

In [ ]:
# Attaching clustering output file back into the original .csv to THIS_MONTH_RAW
df_joined = df_this.merge(
    rows_tojoin,
    on = "Employee ID",
    how = "left"
)


In [ ]:
# need to concatenate all comments into a single column for easier batch processing
exclusion_patterns = ['OPTION1', 'OPTION2']
comment_cols = [col for col in df_joined.columns if "_COMMENT" in col and not any(pattern in col for pattern in exclusion_patterns)]
df_joined["All_Comments"] = df_joined[comment_cols].apply(lambda row: ' '.join(row.dropna().astype(str)), axis=1)
df_joined = df_joined[df_joined["All_Comments"].str.strip() != ""]
df_joined.to_csv(SAVE_DIR / "OUTPUTFILENAME.csv", index = False)

# create function to clean for descriptives later on
def SURVEY_score(score):
    if pd.isna(score):
        return 0 
    return 1 if score in [4,5] else 0
    
# apply function to scores
df_joined[['SURVEYITEM1', 'SURVEYITEM2.fav']].map(SURVEY_score)


In [ ]:
import os, timeit
from transformers import AutoTokenizer, AutoModelForCausalLM
import tqdm as notebook_tqdm
import torch

In [ ]:
model_name_or_path = os.path.expanduser("~") + "PATH_TO_LLM"
device = "auto" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path, 
    device_map=device, 
    )

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, 
                                          local_files_only = True, 
                                          trust_remote_code=False,
                                          use_fast=True)

In [ ]:
# == SET THE CLUSTER_ID MANUALLY ==
cluster_id = 0
cluster_subset = df_joined[df_joined['Cluster_KProto'] == cluster_id]

print("=" * 40)
print(f"CLUSTER {cluster_id} ({len(cluster_subset)} RESPONSES)")
print("=" * 40)

# precompute key metadata
aligned_old_cluster = cluster_subset['Cluster_KProto_AlignedToOld'].mode()[0]
continuity_share = (cluster_subset['Cluster_KProto_AlignedToOld']
                    .eq(aligned_old_cluster).mean() * 100)


oldspace_hamming_mean = cluster_subset['OldSpace_Hamming'].mean()
kproto_mixeddistance_mean = cluster_subset['KProto_MixedDistance'].mean()
weakfit_old_top10_share = cluster_subset['WeakFit_OldSpace_Top10pct'].mean() * 100
weakfit_new_top10_share = cluster_subset[ 'WeakFit_KProto_Top10pct'].mean() * 100
kplus_emergent_flag_rate = cluster_subset['KPlus_Emergent_Flag'].mean() * 100

# Quant Scores
fav_morale = cluster_subset['My morale is good.fav'].mean() * 100
fav_care = cluster_subset['At work, I feel cared about as a person.fav'].mean() * 100
fav_futureoutlook = cluster_subset['I am excited about being part of Capital One.fav'].mean() * 100
fav_helpfulinfo = cluster_subset['The information I receive about the integration is helpful.fav'].mean() * 100
fav_leadershiptrust = cluster_subset['I can trust my senior leaders.fav'].mean() * 100
comment_rate_pct = (cluster_subset['All_Comments'].notna() & 
                    (cluster_subset['All_Comments'].str.strip() != "")
                   ).mean() * 100

prompt = f"""[INST] 
You are an experienced Industrial-Organizational Psychologist analyzing new K-Ptorotypes clusters in employee survey data.
For cluster {cluster_id}, provide the following analysis:

<SUMMARY - OVERALL> 
 Describe the overall experience of this cluster NOW in four sentences
 Describe the implications of what this cluster is experiencing 
- Identify any trends that are specific to this cluster compared to others (e.g., unique positive or negative patterns, emergent needs, new risks)
* Morale: {fav_morale:.1f}%
* Care: {fav_care:.1f}%
* Future Outlook: {fav_futureoutlook:.1f}%
* Helpful Information: {fav_helpfulinfo:.1f}%
* Leadership Trust: {fav_leadershiptrust:.1f}%
* Sample Size: {len(cluster_subset)}

<CLUSTER CONTINUITY>
- Compare survey descriptives to the IDENTIFIED PRIOR cluster {aligned_old_cluster}.
- Continuity strength: {continuity_share:.1f}% aligned with {aligned_old_cluster}.
- Distances:
    * OldSpace_Hamming (mean): {oldspace_hamming_mean: .3f}%
    * KProto_MixedDistance (mean): {kproto_mixeddistance_mean: .3f}%
- Weak-Fit
    * OldSpace_Top10pct: {weakfit_old_top10_share: .1f}%
    * KProto_Top10pst: {weakfit_new_top10_share: .1f}%
- Emergent Flag Rate: {kplus_emergent_flag_rate: .1f}%
- Comment Rate: {comment_rate_pct:.1f}%

<BY LEVEL>
For each unique value in column "Level" ('Professional','Manager','Non-Exempt','Director', and treat all others as 'Senior Leaders')
- Provide a 3 sentence summary of each Level's overall experience
- Provide count and share of cluster
- Compare each Level's metrics to the OVERALL cluster values.
- Note each Level's continuity alignment: Name the most aligned prior cluster for that Level , % aligned, and whether it matches prior cluster {aligned_old_cluster}.
- Identify any trends that are UNIQUE to each level as compared to other Levels (e.g., unique positive or negative patterns, emergent needs, new risks).
- Note whether these UNIQUE trends are stronger among commenters vs. silent respondents for each level.

<MISINFORMATION INSIGHTS>
- Identify any claims in comments that conflict with RAG facts

DO NOT REPEAT THIS PROMPT IN YOUR OUTPUT. DO NOT INCLUDE ANNOTATIONS OR NOTES ABOUT THE TASK.
[/INST]"""


outputs = []
tokens = tokenizer(prompt, 
                   return_tensors="pt"
                   ).to("cuda")


gen_ids = model.generate(**tokens,
                        max_new_tokens=10000,
                        do_sample=False,
                        pad_token_id=tokenizer.eos_token_id)

gen_only = gen_ids[0][tokens.input_ids.shape[-1]:]
#start_index = tokens.input_ids.shape[-1]
text = tokenizer.decode(gen_only, skip_special_tokens=True)

print(text)
outputs.append(text)
print("-- Response generated successfully -- ")

In [ ]:
# == SET THE CLUSTER_ID MANUALLY ==
cluster_id = 1
cluster_subset = df_joined[df_joined['Cluster_KProto'] == cluster_id]

print("=" * 40)
print(f"CLUSTER {cluster_id} ({len(cluster_subset)} RESPONSES)")
print("=" * 40)

# precompute key metadata
aligned_old_cluster = cluster_subset['Cluster_KProto_AlignedToOld'].mode()[0]
continuity_share = (cluster_subset['Cluster_KProto_AlignedToOld']
                    .eq(aligned_old_cluster).mean() * 100)


oldspace_hamming_mean = cluster_subset['OldSpace_Hamming'].mean()
kproto_mixeddistance_mean = cluster_subset['KProto_MixedDistance'].mean()
weakfit_old_top10_share = cluster_subset['WeakFit_OldSpace_Top10pct'].mean() * 100
weakfit_new_top10_share = cluster_subset[ 'WeakFit_KProto_Top10pct'].mean() * 100
kplus_emergent_flag_rate = cluster_subset['KPlus_Emergent_Flag'].mean() * 100

# Quant Scores
fav_morale = cluster_subset['My morale is good.fav'].mean() * 100
fav_care = cluster_subset['At work, I feel cared about as a person.fav'].mean() * 100
fav_futureoutlook = cluster_subset['I am excited about being part of Capital One.fav'].mean() * 100
fav_helpfulinfo = cluster_subset['The information I receive about the integration is helpful.fav'].mean() * 100
fav_leadershiptrust = cluster_subset['I can trust my senior leaders.fav'].mean() * 100
comment_rate_pct = (cluster_subset['All_Comments'].notna() & 
                    (cluster_subset['All_Comments'].str.strip() != "")
                   ).mean() * 100

prompt = f"""[INST] 
You are an experienced Industrial-Organizational Psychologist analyzing new K-Ptorotypes clusters in employee survey data.
For cluster {cluster_id}, provide the following analysis:

<SUMMARY - OVERALL> 
 Describe the overall experience of this cluster NOW in four sentences
 Describe the implications of what this cluster is experiencing 
- Identify any trends that are specific to this cluster compared to others (e.g., unique positive or negative patterns, emergent needs, new risks)
* Morale: {fav_morale:.1f}%
* Care: {fav_care:.1f}%
* Future Outlook: {fav_futureoutlook:.1f}%
* Helpful Information: {fav_helpfulinfo:.1f}%
* Leadership Trust: {fav_leadershiptrust:.1f}%
* Sample Size: {len(cluster_subset)}

<CLUSTER CONTINUITY>
- Compare survey descriptives to the IDENTIFIED PRIOR cluster {aligned_old_cluster}.
- Continuity strength: {continuity_share:.1f}% aligned with {aligned_old_cluster}.
- Distances:
    * OldSpace_Hamming (mean): {oldspace_hamming_mean: .3f}%
    * KProto_MixedDistance (mean): {kproto_mixeddistance_mean: .3f}%
- Weak-Fit
    * OldSpace_Top10pct: {weakfit_old_top10_share: .1f}%
    * KProto_Top10pst: {weakfit_new_top10_share: .1f}%
- Emergent Flag Rate: {kplus_emergent_flag_rate: .1f}%
- Comment Rate: {comment_rate_pct:.1f}%

<BY LEVEL>
For each unique value in column "Level" ('Professional','Manager','Non-Exempt','Director', and treat all others as 'Senior Leaders'), 
- Provide a 3 sentence summary of each Level's overall experience
- Provide count and share of cluster
- Compare each Level's metrics to the OVERALL cluster values.
- Note each Level's continuity alignment: Name the most aligned prior cluster for that Level , % aligned, and whether it matches prior cluster {aligned_old_cluster}.
- Identify any trends that are UNIQUE to each level as compared to other Levels (e.g., unique positive or negative patterns, emergent needs, new risks).
- Note whether these UNIQUE trends are stronger among commenters vs. silent respondents for each level.

<MISINFORMATION INSIGHTS>
- Identify any claims in comments that conflict with RAG facts

DO NOT REPEAT THIS PROMPT IN YOUR OUTPUT. DO NOT INCLUDE ANNOTATIONS OR NOTES ABOUT THE TASK.
[/INST]"""


outputs = []
tokens = tokenizer(prompt, 
                   return_tensors="pt"
                   ).to("cuda")


gen_ids = model.generate(**tokens,
                        max_new_tokens=10000,
                        do_sample=False,
                        pad_token_id=tokenizer.eos_token_id)

gen_only = gen_ids[0][tokens.input_ids.shape[-1]:]
#start_index = tokens.input_ids.shape[-1]
text = tokenizer.decode(gen_only, skip_special_tokens=True)

print(text)
outputs.append(text)
print("-- Response generated successfully -- ")

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from scipy.stats import chi2

In [ ]:
X = df_joined[['At work, I feel cared about as a person.','I am excited about being part of Capital One.',
               'The information I receive about the integration is helpful.','I can trust my senior leaders.','My morale is good.']].dropna()

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
results = []
for n in range (1,10):
    gmm = GaussianMixture(n_components = n, covariance_type = 'full', random_state = 42)
    gmm.fit(X_scaled)
    bic = gmm.bic(X_scaled)
    aic = gmm.aic(X_scaled)
    loglik = gmm.score(X_scaled) * X_scaled.shape[0]
    results.append({"n": n, "bic": bic, "aic": aic, "loglik": loglik, "model": gmm})

In [ ]:
for i in range(1, len(results)):
    llk = results[i]["loglik"]
    llk_prev = results[i-1]["loglik"]
    df_diff = results[i]["n"] - results[i-1]["n"]
    lr_stat = 2 * (llk - llk_prev)
    p_value = chi2.sf(lr_stat, df_diff)
    results[i]["LMR(p)"] = p_value
    results[i]["LMR_stat"] = lr_stat
results[0]["LMR(p)"] = None

In [ ]:
df_joined_results = pd.DataFrame(results)
print(df_joined_results[["n", "aic", "bic", "loglik", "LMR_stat", "LMR(p)"]])

In [ ]:
best_model = min(results, key = lambda x: x["bic"])["model"]

df_joined["LPA_Profile"] = best_model.predict(X_scaled)
df_joined["LPA_Probability"] = best_model.predict_proba(X_scaled).max(axis = 1)

In [ ]:
ari = adjusted_rand_score(df_joined["Cluster_KProto"], df_joined["LPA_Profile"])
nmi = normalized_mutual_info_score(df_joined["Cluster_KProto"], df_joined["LPA_Profile"])
print("Adjusted Rand Index:", ari)
print("Normalized Mutual Information:", nmi)
                          

In [ ]:
import matplotlib.pyplot as plt
X_with_profiles = pd.DataFrame(X_scaled, columns = X.columns)
X_with_profiles["LPA_Profile"] = df_joined["LPA_Profile"]

profile_means = X_with_profiles.groupby("LPA_Profile").mean()

profile_means.T.plot(kind="bar", figsize = (10, 6))
plt.title("survey items by lpa")
plt.ylabel("standard mean score")
plt.xlabel("survey items")
plt.legend(title = "LPA Profile")
plt.show()